<a id="optional-fno-ablation"></a>
# 선택 실습 — FNO 푸리에 모드 수와 계산 비용·오차 비교

이 노트북은 [Poisson FNO 필수 실습](../02_Poisson_FNO.ipynb)을 마친 뒤 강사 안내에 따라 진행합니다. 필수 학습과 동시에 실행하지 않습니다.

**비교 범위:** 데이터와 학습 조건을 고정하고 `fno_modes`만 6에서 12로 늘려 실행 시간, 파라미터 수, GPU 메모리, 테스트 오차의 실제 차이를 측정합니다.

두 실험은 동일한 축소 데이터셋과 난수 시드, 모델 층 수, 채널 너비, 배치 크기, 200단계 학습을 사용합니다. 바꾸는 요인은 `fno_modes` 하나입니다.


## 1. 환경과 통제 조건 확인

두 설정 파일을 비교해 `fno_modes`와 결과 이름 외의 조건이 같은지 검사합니다.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys
import time

launch_dir = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (launch_dir, *launch_dir.parents)
        if (candidate / "labs" / "poisson_fno" / "train_fno.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("labs/poisson_fno/train_fno.py를 찾지 못했습니다.")

LAB_DIR = REPO_ROOT / "labs" / "poisson_fno"
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

import torch
import physicsnemo
import physicsnemo.sym
from notebook_utils import assert_ablation_configs_match, ensure_profile_dataset

config_6 = LAB_DIR / "conf" / "config_FNO_ablation_6.yaml"
config_12 = LAB_DIR / "conf" / "config_FNO_ablation_12.yaml"
differences = assert_ablation_configs_match(config_6, config_12)

print("과정 루트     : {}".format(REPO_ROOT))
print("FNO 실습 폴더 : {}".format(LAB_DIR))
print("GPU           : {}".format(
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA 사용 불가"
))
print("통제한 설정 차이:")
for key, values in differences.items():
    print("  {}: {!r} → {!r}".format(key, values[0], values[1]))


## 2. 비교 기준

데이터의 `max_mode=6`과 모델의 `fno_modes=6/12`는 서로 다른 설정입니다. 전자는 데이터에 포함한 최고 주파수, 후자는 모델이 처리하는 푸리에 모드 수입니다. 아래 실행은 두 설정의 측정값과 12/6 비율을 자동으로 계산합니다.


## 3. 공통 축소 데이터셋 준비

두 실험은 같은 64×64 학습·검증·테스트 데이터를 재사용합니다. 파일 검증을 통과하면 새로 생성하지 않습니다.


In [ ]:
PROFILE_INFO = ensure_profile_dataset(
    LAB_DIR,
    "recovery",
    force=False,
    device="cuda" if torch.cuda.is_available() else "auto",
)
print("공유 데이터셋: {}".format(PROFILE_INFO["dataset_dir"]))


## 4. 모드 6개와 12개를 차례로 실행

두 실행 사이에 바꾸는 것은 `fno_modes` 하나뿐입니다. 데이터, 난수 시드, 층 수, 채널 너비, 배치 크기, 학습 단계(200)는 모두 같습니다. 위 셀의 설정 비교 검사가 이를 이미 확인했습니다.

각 실행의 전체 시간에는 CUDA 초기화와 파일 입출력이 포함됩니다.


In [ ]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
experiments = [
    {"modes": 6, "config": "config_FNO_ablation_6"},
    {"modes": 12, "config": "config_FNO_ablation_12"},
]
RESULTS = []

for experiment in experiments:
    modes = experiment["modes"]
    output_dir = LAB_DIR / "outputs" / "ksc_fno_ablation" / RUN_ID / "modes_{}".format(modes)
    metrics_path = output_dir / "final_state_test_metrics.json"
    command = [
        sys.executable,
        "train_fno.py",
        "--config-name",
        experiment["config"],
        "network_dir={}".format(output_dir),
        "custom.metrics_file={}".format(metrics_path),
    ]

    print("\n" + "=" * 72)
    print("FNO modes={} 실행".format(modes))
    started = time.perf_counter()
    completed = subprocess.run(command, cwd=LAB_DIR)
    wall_seconds = time.perf_counter() - started
    if completed.returncode != 0:
        raise RuntimeError("modes={} 학습 실패 (exit code={})".format(modes, completed.returncode))

    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    assert payload["profile"] == "recovery"
    assert payload["random_seed"] == 2026
    assert payload["model"]["fno_modes"] == modes
    RESULTS.append(
        {
            "modes": modes,
            "parameters": payload["model"]["trainable_parameters"],
            "wall_seconds": wall_seconds,
            "peak_memory_bytes": payload["runtime_observation"]["peak_memory_allocated_bytes"],
            "relative_l2_before": payload["metrics_before_training"]["relative_l2"],
            "relative_l2_after": payload["metrics_after_training"]["relative_l2"],
            "rmse_after": payload["metrics_after_training"]["rmse"],
        }
    )

print("\n두 실험이 완료되었습니다.")


## 5. 결과 비교

모델 크기·실행 시간·메모리는 계산 비용을 서로 다른 관점에서 보여 줍니다. 상대 L2 오차는 작을수록 예측이 정답에 가깝습니다.


In [ ]:
import matplotlib.pyplot as plt

header = "{:>7} {:>14} {:>12} {:>11} {:>14} {:>14}".format(
    "Modes", "Parameters", "Wall min", "Peak GiB", "L2 before", "L2 after"
)
print(header)
for result in RESULTS:
    peak_gib = (
        result["peak_memory_bytes"] / 2**30
        if result["peak_memory_bytes"] is not None
        else float("nan")
    )
    print("{:7d} {:14,d} {:12.2f} {:11.2f} {:14.6e} {:14.6e}".format(
        result["modes"],
        result["parameters"],
        result["wall_seconds"] / 60.0,
        peak_gib,
        result["relative_l2_before"],
        result["relative_l2_after"],
    ))

low, high = RESULTS
ratios = {
    "parameter_ratio_12_over_6": high["parameters"] / low["parameters"],
    "wall_time_ratio_12_over_6": high["wall_seconds"] / low["wall_seconds"],
    "error_ratio_12_over_6": high["relative_l2_after"] / low["relative_l2_after"],
}
if low["peak_memory_bytes"] is not None and high["peak_memory_bytes"] is not None:
    ratios["peak_memory_ratio_12_over_6"] = (
        high["peak_memory_bytes"] / low["peak_memory_bytes"]
    )
print("\n비율(modes=12 / modes=6)")
for name, value in ratios.items():
    print("  {:32s}: {:.3f}".format(name, value))

labels = ["modes={}".format(result["modes"]) for result in RESULTS]
figure, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
axes[0].bar(labels, [r["wall_seconds"] / 60.0 for r in RESULTS], color=["#76B900", "#1f6feb"])
axes[0].set(title="전체 실행 시간", ylabel="분")
axes[1].bar(labels, [r["parameters"] for r in RESULTS], color=["#76B900", "#1f6feb"])
axes[1].set(title="학습 파라미터 수", ylabel="개")
axes[2].bar(labels, [r["relative_l2_after"] for r in RESULTS], color=["#76B900", "#1f6feb"])
axes[2].set(title="학습 후 테스트 오차", ylabel="상대 L2")
plt.show()


In [ ]:
error_ratio = ratios["error_ratio_12_over_6"]
print("자동 비교 요약")
print("파라미터 수 12/6 : {:.3f}×".format(ratios["parameter_ratio_12_over_6"]))
print("전체 시간 12/6   : {:.3f}×".format(ratios["wall_time_ratio_12_over_6"]))
if "peak_memory_ratio_12_over_6" in ratios:
    print("최대 메모리 12/6 : {:.3f}×".format(ratios["peak_memory_ratio_12_over_6"]))
else:
    print("최대 메모리 12/6 : 측정값 없음")
print("테스트 오차 12/6 : {:.3f}×".format(error_ratio))
if error_ratio < 1.0:
    print("modes=12의 테스트 오차가 {:.1f}% 낮습니다.".format((1.0 - error_ratio) * 100.0))
elif error_ratio > 1.0:
    print("modes=12의 테스트 오차가 {:.1f}% 높습니다.".format((error_ratio - 1.0) * 100.0))
else:
    print("두 실행의 테스트 오차가 같습니다.")


## 6. 해석 범위

**강사와 함께 확인할 지점**

- 파라미터 수·시간·메모리·테스트 오차의 12/6 비율을 나란히 봅니다. 비용이 두 배 늘 때 오차도 그만큼 줄었습니까?
- 이 데이터의 최고 주파수는 `max_mode=6`입니다. 모델이 12개 모드를 처리해도 데이터에 없는 주파수까지 배울 것은 없습니다. 위 결과가 이 예상과 맞습니까?

**이 결과의 해석 범위**

지정된 recovery 데이터셋, 난수 시드 하나, 200단계 학습, 지금 이 노드 상태에서 얻은 관찰값입니다. 실행 순서를 무작위화하거나 반복하지 않았으므로 작은 시간 차이는 의미 있는 성능 차이로 읽지 않습니다. 일반적인 결론에는 여러 시드의 반복 실행과 그 평균·분산이 필요합니다.


---

## 참고 자료와 라이선스

공식 문서와 논문 링크는 [PhysicsNeMo 모듈 안내](../README.md#참고-자료)에 모았습니다. 노트북 실행에는 인터넷 연결이 필요하지 않습니다.

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). Existing file-level notices remain in effect.
